# F-003-3e: YOLO-Fastest 人物検出モデル学習 (Ethos-U55 NPU対応)

Ethos-U55 NPUで動作実績のあるYOLO-Fastest (YOLOv3ベース, アンカーベース) アーキテクチャで
人物検出モデルを学習し、TFLite INT8に変換する。

## 背景 (Issue #126)
- YOLOv8カスタムモデルはEthos-U55 NPU上でクラススコアが全て0になり検出不能
- 原因: YOLOv8のStridedSliceオペレータがNPU未対応→3サブグラフ分割
- YOLO-Fastestは全オペレータがNPU対応→単一サブグラフで実行可能

## モデル構成
- ベース: Nota-NetsPresso/ModelZoo-YOLOFastest-for-ARM-U55-M85
- 入力: 192x192x1 (Grayscale, INT8)
- 出力: 2ブランチ (6x6 + 12x12, 各3アンカー)
- クラス: 1 (person)
- 検出方式: アンカーベース (YOLOv3スタイル)

## 前提条件
- Google Colab (GPU: T4推奨)
- Google Driveに `fall_detection_dataset.zip` をアップロード済み

## ワークフロー
1. GPU確認・環境構築
2. データセット準備 (Grayscale変換)
3. モデル設定 (YOLO-Fastest person, 192x192x1)
4. モデル学習
5. 精度評価
6. ONNX エクスポート
7. TFLite INT8 変換
8. 成果物ダウンロード

---
## Step 1: GPU確認・環境構築

**重要:** メニューの「ランタイム > ランタイムのタイプを変更」で **GPU (T4)** を選択してください。

### 注意
- このリポジトリは PyTorch < 2.0 を要求します
- pip installセル実行後、**ランタイムを再起動**してください

In [ ]:
# GPU 確認
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
    print('=== GPU が利用可能です ===')
else:
    print('WARNING: GPU が検出されません。ランタイムを GPU に変更してください。')

In [ ]:
# YOLO-Fastest リポジトリのセットアップ
# ColabデフォルトのPyTorch 2.x をそのまま使用
# (リポジトリはPyTorch < 2.0を要求するが、実際には動作する)

!pip install -q numpy==1.26.4

# YOLO-Fastest リポジトリをクローン
!git clone https://github.com/Nota-NetsPresso/ModelZoo-YOLOFastest-for-ARM-U55-M85.git /content/yolo-fastest

# 依存パッケージ (PyTorchバージョン制約を緩和)
!cd /content/yolo-fastest && sed -i 's/torch>=.*/torch/' requirements.txt && pip install -q -r requirements.txt
!pip install -q onnx onnx2tf onnxsim

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
print('=== セットアップ完了 ===')

---
## Step 2: データセット準備 (Grayscale変換)

### 事前準備 (ローカルPCで実行)

```bash
cd mimamori-sense/dataset/merged
zip -r fall_detection_dataset.zip images/ labels/
```

作成した `fall_detection_dataset.zip` を Google Drive のマイドライブ直下にアップロードしてください。

In [ ]:
# Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

WORK_DIR = '/content/yolo-fastest'
DATASET_DIR = os.path.join(WORK_DIR, 'dataset')
DATASET_GRAY_DIR = os.path.join(WORK_DIR, 'dataset_gray')
DATASET_ZIP = '/content/drive/MyDrive/fall_detection_dataset.zip'

os.chdir(WORK_DIR)
print(f'作業ディレクトリ: {WORK_DIR}')

# データセット展開
if not os.path.isdir(DATASET_DIR):
    if os.path.isfile(DATASET_ZIP):
        print('データセット展開中...')
        !mkdir -p {DATASET_DIR} && unzip -q {DATASET_ZIP} -d {DATASET_DIR}
        print('展開完了')
    else:
        print(f'ERROR: {DATASET_ZIP} が見つかりません')
else:
    print('データセットは展開済みです')

# 検証
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DATASET_DIR, 'images', split)
    lbl_dir = os.path.join(DATASET_DIR, 'labels', split)
    if os.path.isdir(img_dir):
        img_count = len([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
        lbl_count = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')]) if os.path.isdir(lbl_dir) else 0
        print(f'  {split}: images={img_count}, labels={lbl_count}')

In [ ]:
# Grayscale変換 (NPU互換: 1ch入力)
from PIL import Image
import shutil

print('=== Grayscale データセット作成 ===')

converted_count = 0
for split in ['train', 'val', 'test']:
    src_img_dir = os.path.join(DATASET_DIR, 'images', split)
    dst_img_dir = os.path.join(DATASET_GRAY_DIR, 'images', split)
    os.makedirs(dst_img_dir, exist_ok=True)

    src_lbl_dir = os.path.join(DATASET_DIR, 'labels', split)
    dst_lbl_dir = os.path.join(DATASET_GRAY_DIR, 'labels', split)
    os.makedirs(dst_lbl_dir, exist_ok=True)

    if not os.path.isdir(src_img_dir):
        continue

    img_files = [f for f in os.listdir(src_img_dir)
                 if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    for img_file in img_files:
        src_path = os.path.join(src_img_dir, img_file)
        dst_path = os.path.join(dst_img_dir, img_file)
        try:
            img = Image.open(src_path).convert('L')
            img.save(dst_path)
            converted_count += 1
        except Exception as e:
            print(f'  ERROR: {img_file}: {e}')

    if os.path.isdir(src_lbl_dir):
        for lbl_file in os.listdir(src_lbl_dir):
            if lbl_file.endswith('.txt'):
                shutil.copy2(os.path.join(src_lbl_dir, lbl_file),
                             os.path.join(dst_lbl_dir, lbl_file))

    img_count = len([f for f in os.listdir(dst_img_dir)
                     if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
    print(f'  {split}: {img_count} images')

print(f'\n合計 {converted_count} 枚の画像をGrayscaleに変換しました')

In [ ]:
# Grayscale変換はスキップ (RGB 3chで学習)
# YOLOv5データローダーがRGB固定のため、RGBデータセットをそのまま使用
print('=== RGB データセット使用 (Grayscale変換スキップ) ===')
print('YOLOv5データローダーがRGB固定のため、RGBで学習します。')
print()
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DATASET_DIR, 'images', split)
    if os.path.isdir(img_dir):
        img_count = len([f for f in os.listdir(img_dir)
                         if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
        print(f'  {split}: {img_count} images')

In [ ]:
# data.yaml 作成 (person検出, 1クラス, RGBデータセット)
data_yaml_content = f"""path: {DATASET_DIR}
train: images/train
val: images/val
test: images/test

nc: 1
names: ['person']
"""

data_yaml_path = os.path.join(WORK_DIR, 'data', 'person_fall.yaml')
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f'data.yaml 作成完了: {data_yaml_path}')
print(data_yaml_content)

In [ ]:
import yaml

# 元の yolo-fastest.yaml を読み込み
base_yaml_path = os.path.join(WORK_DIR, 'models', 'yolo-fastest.yaml')
with open(base_yaml_path, 'r') as f:
    model_cfg = yaml.safe_load(f)

# 人物検出用に修正
model_cfg['nc'] = 1           # person only
# ch はデフォルト3 (RGB) のまま。YOLOv5データローダーがRGB固定のため。

# 保存
person_yaml_path = os.path.join(WORK_DIR, 'models', 'yolo-fastest-person.yaml')
with open(person_yaml_path, 'w') as f:
    yaml.dump(model_cfg, f, default_flow_style=None, sort_keys=False)

print(f'モデル設定: {person_yaml_path}')
print(f'  nc: {model_cfg["nc"]}')
print(f'  ch: {model_cfg.get("ch", 3)} (RGB)')
print(f'  anchors: {model_cfg["anchors"]}')

In [ ]:
# 学習パラメータ
IMG_SIZE = 192
EPOCHS = 200
BATCH_SIZE = 64

print(f'=== 学習設定 ===')
print(f'  入力: {IMG_SIZE}x{IMG_SIZE}x1 (Grayscale)')
print(f'  エポック: {EPOCHS}')
print(f'  バッチサイズ: {BATCH_SIZE}')
print(f'  クラス: person (1クラス)')

---
## Step 4: モデル学習

YOLO-Fastest をスクラッチ学習する。

### 接続切れからの再開
学習結果はGoogle Driveに自動保存されます。
接続が切れた場合:
1. ランタイムを再起動
2. Step 1 (GPUセル以外) から Step 3 まで順に再実行
3. Step 4 のセルを実行

### 再学習する場合
```python
!rm -rf /content/drive/MyDrive/yolo_fastest_person/train
```

In [ ]:
import os

GDRIVE_TRAIN_DIR = '/content/drive/MyDrive/yolo_fastest_person'
os.makedirs(GDRIVE_TRAIN_DIR, exist_ok=True)

# 前回の学習が中断された場合、last.pt から再開
last_pt = os.path.join(GDRIVE_TRAIN_DIR, 'train', 'weights', 'last.pt')

os.chdir(WORK_DIR)

if os.path.exists(last_pt):
    print('=== 前回の学習を再開 ===')
    print(f'再開ポイント: {last_pt}')
    !python train.py --resume {last_pt}
else:
    print('=== 新規学習開始 ===')
    !python train.py \
        --weights '' \
        --cfg ./models/yolo-fastest-person.yaml \
        --data ./data/person_fall.yaml \
        --epochs {EPOCHS} \
        --imgsz {IMG_SIZE} \
        --batch-size {BATCH_SIZE} \
        --device 0 \
        --workers 2 \
        --patience 50 \
        --project {GDRIVE_TRAIN_DIR} \
        --name train \
        --exist-ok

print('\n=== 学習完了 ===')

In [ ]:
# 学習曲線の表示
from IPython.display import Image, display
import os

results_png = os.path.join(GDRIVE_TRAIN_DIR, 'train', 'results.png')
if os.path.exists(results_png):
    display(Image(filename=results_png, width=800))
else:
    print('学習結果の画像が見つかりません')

---
## Step 5: 精度評価

In [ ]:
import os

best_pt = os.path.join(GDRIVE_TRAIN_DIR, 'train', 'weights', 'best.pt')
os.chdir(WORK_DIR)

!python val.py \
    --weights {best_pt} \
    --data ./data/person_fall.yaml \
    --imgsz {IMG_SIZE} \
    --batch-size {BATCH_SIZE} \
    --device 0 \
    --task val

print('\n=== 精度評価完了 ===')

---
## Step 6: ONNX エクスポート

In [ ]:
import os

os.chdir(WORK_DIR)

ONNX_PATH = os.path.join(WORK_DIR, 'yolo_fastest_person.onnx')

!python export.py \
    --weights {best_pt} \
    --imgsz {IMG_SIZE} \
    --include onnx \
    --opset 11 \
    --simplify

# export.py は best.pt と同じディレクトリに best.onnx を生成
exported_onnx = best_pt.replace('.pt', '.onnx')
if os.path.exists(exported_onnx):
    import shutil
    shutil.move(exported_onnx, ONNX_PATH)
    print(f'ONNX エクスポート完了: {ONNX_PATH}')
    print(f'サイズ: {os.path.getsize(ONNX_PATH)/1024:.1f} KB')
else:
    # export.pyが別の場所に出力する場合
    import glob
    onnx_files = glob.glob(os.path.join(GDRIVE_TRAIN_DIR, '**/*.onnx'), recursive=True)
    if onnx_files:
        shutil.move(onnx_files[0], ONNX_PATH)
        print(f'ONNX エクスポート完了: {ONNX_PATH}')
    else:
        print('ERROR: ONNX ファイルが見つかりません')

---
## Step 7: TFLite INT8 変換

In [ ]:
import numpy as np
import glob
from PIL import Image
import os

SAVED_MODEL_DIR = os.path.join(WORK_DIR, 'saved_model')
FP32_PATH = os.path.join(WORK_DIR, 'yolo_fastest_person_fp32.tflite')
INT8_PATH = os.path.join(WORK_DIR, 'yolo_fastest_person_int8.tflite')

# ONNX -> SavedModel -> TFLite FP32
print('=== ONNX -> SavedModel (onnx2tf) ===')
!onnx2tf -i {ONNX_PATH} -o {SAVED_MODEL_DIR} -osd 2>&1 | tail -10

if os.path.isdir(SAVED_MODEL_DIR):
    import tensorflow as tf

    # FP32 TFLite
    print('\n=== FP32 TFLite 変換 ===')
    converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
    tflite_fp32 = converter.convert()
    with open(FP32_PATH, 'wb') as f:
        f.write(tflite_fp32)
    print(f'FP32 TFLite: {os.path.getsize(FP32_PATH)/1024:.1f} KB')

    # INT8 量子化 (Grayscale)
    print('\n=== INT8 量子化 (Grayscale) ===')
    cal_dir = os.path.join(DATASET_GRAY_DIR, 'images', 'val')
    cal_images = sorted(glob.glob(os.path.join(cal_dir, '*.jpg')))[:200]
    if not cal_images:
        cal_images = sorted(glob.glob(os.path.join(cal_dir, '*.png')))[:200]
    print(f'キャリブレーション画像: {len(cal_images)}枚')

    # FP32モデルの入力形状を取得
    interp = tf.lite.Interpreter(model_path=FP32_PATH)
    interp.allocate_tensors()
    inp_detail = interp.get_input_details()[0]
    inp_shape = inp_detail['shape']
    n, h, w, c = inp_shape
    print(f'入力形状: {inp_shape} (NHWC), ch={c}')

    def representative_dataset():
        for img_path in cal_images:
            img = Image.open(img_path).convert('L').resize((w, h))
            arr = np.array(img, dtype=np.float32) / 255.0
            if c == 1:
                arr = arr.reshape(1, h, w, 1)
            else:
                arr = np.stack([arr]*3, axis=-1).reshape(1, h, w, 3)
            yield [arr]

    converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    try:
        int8_model = converter.convert()
        with open(INT8_PATH, 'wb') as f:
            f.write(int8_model)
        int8_kb = os.path.getsize(INT8_PATH) / 1024
        print(f'INT8 TFLite: {int8_kb:.1f} KB')
    except Exception as e:
        print(f'INT8 量子化エラー: {e}')
else:
    print('ERROR: SavedModel の生成に失敗しました')

In [ ]:
# モデル詳細確認
import tensorflow as tf
import numpy as np
import os

if os.path.exists(INT8_PATH):
    int8_kb = os.path.getsize(INT8_PATH) / 1024
    print(f'=== INT8 モデル詳細 ===')
    print(f'ファイルサイズ: {int8_kb:.1f} KB')

    interp = tf.lite.Interpreter(model_path=INT8_PATH)
    interp.allocate_tensors()

    for tag, details in [('Input', interp.get_input_details()),
                         ('Output', interp.get_output_details())]:
        print(f'\n--- {tag} ---')
        for i, d in enumerate(details):
            print(f'  [{i}] {d["name"]} shape={d["shape"]} dtype={d["dtype"]}')
            qp = d.get('quantization_parameters', {})
            sc = qp.get('scales', np.array([]))
            zp = qp.get('zero_points', np.array([]))
            if len(sc) > 0:
                print(f'      scale={sc[0]:.8f}, zero_point={zp[0]}')

    print(f'\n--- 重要: 出力テンソルの scale/zero_point を控えてください ---')
    print('MCU後処理コードに設定が必要です')
else:
    print('INT8 モデルが見つかりません')

---
## Step 8: 成果物ダウンロード

In [ ]:
# Google Drive に保存
import shutil
import os

OUTPUT_DIR = '/content/drive/MyDrive/yolo_fastest_person_model'
os.makedirs(OUTPUT_DIR, exist_ok=True)

files_to_copy = {
    best_pt: 'best_person.pt',
    ONNX_PATH: 'yolo_fastest_person.onnx',
    FP32_PATH: 'yolo_fastest_person_fp32.tflite',
    INT8_PATH: 'yolo_fastest_person_int8.tflite',
}

# アンカー情報も保存
anchors_json = os.path.join(GDRIVE_TRAIN_DIR, 'train', 'weights', 'anchors.json')
if os.path.exists(anchors_json):
    files_to_copy[anchors_json] = 'anchors.json'

# 学習曲線
results_png = os.path.join(GDRIVE_TRAIN_DIR, 'train', 'results.png')
if os.path.exists(results_png):
    files_to_copy[results_png] = 'results.png'

for src, dst_name in files_to_copy.items():
    if os.path.exists(src):
        dst = os.path.join(OUTPUT_DIR, dst_name)
        shutil.copy2(src, dst)
        size_kb = os.path.getsize(src) / 1024
        print(f'  {dst_name}: {size_kb:.1f} KB')

print(f'\n=== Google Drive に保存完了: {OUTPUT_DIR} ===')

In [ ]:
# INT8 TFLite モデルをダウンロード
from google.colab import files

if os.path.exists(INT8_PATH):
    files.download(INT8_PATH)
    print('INT8モデルのダウンロードを開始しました')
else:
    print('INT8モデルが見つかりません。Step 7 を先に実行してください。')

---
## まとめ

### 次のステップ (実機デプロイ)

1. INT8 TFLiteモデルをダウンロード
2. `scripts/deploy_fall_detection.ps1` でMERA変換
3. **MERA変換後に単一サブグラフ (sub_0000のみ) であることを確認**
   - `sub_0001`, `sub_0002` が生成されなければ成功
4. 後処理コードをYOLOv3アンカーベースに書き換え
5. e2studioビルド → 実機書き込み → 動作確認

### 備考
- 入力: 192x192x1 Grayscale (顔検出モデルと同じ)
- 出力: 2ブランチ (6x6 + 12x12, 各3アンカー)
- 後処理: YOLOv3アンカーベースデコード (リファレンスDetectorPostProcessing.cc参考)
- 出力テンソルのscale/zero_pointを後処理コードに設定する必要あり